# System One Modelとは

**System One model** は TypeSafe AI が提唱する、ソフトウェアが直接使える「速くて構造化された判断」を返すAIモデルのクラス。

テキストを生成するLLMとは違い、**状態（state）** と **型付きの質問（typed questions）** を受け取り、**型付きの答えと確率** を返す。

:::{card} 参考
- [Introducing System One Models & Jev - TypeSafe AI Blog](https://typesafe.ai/blog/introducing-system-one-models-and-jev)
- [System One - TypeSafe AI Docs](https://docs.typesafe.ai/concepts/system-one)
:::

## 名前の由来

Daniel Kahneman『ファスト&スロー（Thinking, Fast and Slow）』の二重過程理論から来ている。

| | System 1 | System 2 |
|---|---|---|
| 人間の思考 | 速い・直感的・自動的 | 遅い・熟慮的・論理的 |
| AIでの対応 | System One model（Jevなど） | 推論型LLM（Chain-of-Thought等） |

LLMは推論（System 2的な思考）の方向へ進化してきたが、ソフトウェアの自動化で大量に必要になるのは「このチケットは返金依頼か？」「どの部署に回すべきか？」のような **小さく速い判断** であり、それを専門に担うのが System One model という位置づけ。

## Why System One?

創業者の Diogo Almeida（OpenAIで instruction following の開発に携わった）の問題意識は「チャットでは超人的な性能なのに、**自動化はどこにあるのか？**」というもの。

LLMをソフトウェアの部品として使うときの典型的な問題：

- 出力が自由なテキストなので、パースが必要で、JSONが壊れる・ありえない選択肢を返すなどの **型エラー** が起こる
- 自己回帰で1トークンずつ生成するため **遅く、高い**
- RLHFで人間の好みに最適化されているため、**過信（overconfident）** しがちで、確信度が当てにならない（sycophancyや自信満々のハルシネーション）

TypeSafeは「将来の大規模なAI自動化は 99% が機械対機械、1% が人間とのやり取りになる」と考えており、人間に読みやすいテキストよりも **信頼できる制約された出力** が重要だとしている。

## LLMとの違い

| | LLM | System One model |
|---|---|---|
| 出力 | 自由なテキスト | 事前定義された型付きの値（+確率・確信度） |
| 生成方式 | 自己回帰（逐次） | 並列（1回のパスで全質問に回答） |
| 学習目標 | 人間の好み（RLHF）・検証可能な報酬（RLVR） | 結果に対してキャリブレーションされた確率（RLCD） |
| 型エラー | 起こりうる | 構造上起こらない |
| できないこと | － | コード・説明文などのテキスト生成 |

System One model は **テキスト生成能力を捨てる** 代わりに、速度・コスト・信頼性を得ている。

## 基本的な使い方の考え方

1. 判断に必要な文脈を **state** にまとめる（テキスト・JSONオブジェクト・テキストの配列）
2. 独立した **複数の質問** を同時に投げる
3. 返ってきた答えと **confidence** をもとに、通常の（決定的な）コードで分岐する

つまり、LLMにワークフロー全体を任せるのではなく、**制御はコードが持ち、曖昧な判断だけをモデルに任せる** という設計思想。

### 質問の型（primitives）

| 型 | 問い | 返り値 | 例 |
|---|---|---|---|
| Choice | どの選択肢か？（順序なし） | `choice`, `probabilities`, `confidence` | チケットの振り分け、文書分類 |
| Score | どのレベルか？（順序あり） | `score`, `legend`, `probabilities`, `confidence` | バグの深刻度、顧客の苛立ち度 |
| Noul | これは真か？ | `noul`（0〜1の確率） | 返金依頼か、PIIを含むか |

Noulの0.5は「Yes/Noが半々」という意味で、「中くらい」ではない。程度を測りたいならScoreを使う。

## Probability と Confidence

- **probability**：選択肢（レベル）上の確率分布
- **confidence**：その分布がどれだけ1点に集中しているかを 0〜1 に要約した値

1つの選択肢に全確率が乗れば 1.0、一様に広がるほど低くなる。例えば3択のChoiceでは

$$
\text{confidence} = \frac{3 \cdot \max_k p_k - 1}{2}
$$

（一般には $K$ 択なら $\frac{K \max_k p_k - 1}{K - 1}$ と同じ形と思われる）

### Confidenceによる3段階の運用

| confidence | アクション |
|---|---|
| 高 | そのまま自動で実行 |
| 中 | 追加情報を集める・ユーザーに確認する |
| 低 | 人間にエスカレーション・別システムにフォールバック |

閾値は **間違えたときの影響の大きさ** に応じて変える（読み取りだけの操作は低め、送金などの破壊的な操作は高め）。

## キャリブレーション

キャリブレーションされたモデルでは、確率0.2と言った事象が実際に約20%で起き、0.8と言った事象が約80%で起きる。

これにより「確率が高いときは自動実行、低いときは人間へ」という判断を **確率そのものに基づいて** 設計できる。

TypeSafeはこのための事後学習手法を **RLCD（Reinforcement Learning for Calibrated Decisions）** と呼び、RLHF・RLVRに続く第3の事後学習パラダイムと位置付けている。

## 代表的な設計パターン

TypeSafeのドキュメントでは以下のようなパターンが紹介されている（[Patterns](https://docs.typesafe.ai/patterns)）。

- **Speculative Fan-Out**：必要になるかもしれない質問も含めて1回のリクエストでまとめて投げ、コード側で使う答えを選ぶ（並列処理なので質問を増やしてもほぼコストが増えない）
- **Confidence-Gated Routing**：confidenceを分岐の軸にする
- **Composite Scoring**：複雑な判断を小さなScoreに分解し、コードで重み付けして合成する
- **Intent Routing**：リクエストを分類して最適なハンドラに振り分ける

同じリクエスト内の質問は **互いに独立** で、ある答えを別の質問の文脈にはできない。依存関係がある場合はリクエストを分ける。

## エコシステム

- **Jev**：TypeSafe AIの最初のSystem One model（商用API）
- **OpenJev系**：Jevの発表直後に大量に登場したオープンソースの再現・代替実装
- **CLM（Contrastive Language Model）**：対照学習でstateとactionを結ぶ、オープンウェイトのSystem One model